# Wave 2 — Part 2: Post-EDA Modeling Preparation
## Capstone Project — Student High School Dropout Prediction

**Dataset:** HSLS:09 (High School Longitudinal Study of 2009) — Wave 2 (11th Grade Follow-up)

**Purpose:** This notebook picks up after EDA. It applies one-hot encoding to nominal variables, creates the missingness flag, drops the STU_ID, and exports four model-ready datasets.


---
# Section 1 — Load Data and Rebuild Variables



The cell below:
1. imports all libraries
2. Loads `Wave2_EDA_Ready.csv` (the cleaned, pre-encoding snapshot)
3. Loads `Wave1_FeatureSelected.csv` (needed for Dataset 3 export)
4. Rebuilds all column-list variables that were defined during cleaning in Part 1

> **Note:** Column lists (`cleaned_continuous_cols_w2`, `cleaned_ordinal_cols_w2`, `cleaned_binary_cols_w2`) are hard-coded here to match exactly what Part 1 retained after the 90%-validity filter.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer

# ── 1. Load the EDA-ready dataset (output of Part 1) ─────────────────────────
wave2 = pd.read_csv("Wave2_EDA_Ready.csv")
print(f"Loaded Wave2_EDA_Ready.csv  — shape: {wave2.shape}")

# ── 2. Load Wave 1 feature-selected dataset (needed for Dataset 3 export) ────
wave1_selected = pd.read_csv("Wave1_FeatureSelected.csv")
print(f"Loaded Wave1_FeatureSelected.csv — shape: {wave1_selected.shape}")

# ── 3. Rebuild column lists (mirror Part 1 cleaning results) ─────────────────
# These reflect the columns that PASSED the 90%-validity filter in Part 1.

cleaned_continuous_cols_w2 = [
    'X2MTHEFF',
    'X2SCIEFF',
    'X2TXMTSCOR',
    'X2BEHAVEIN',
    'X2PROBLEM',
    # School-level ACT/SAT averages — verify these passed the 90% threshold in Part 1
    # and add/remove as needed based on Part 1 output
    'C2AVGACTENG',
    'C2AVGACTMATH',
    'C2AVGACTSCI',
    'C2AVGACTCOMP',
    'C2AVGACTREAD',
    'C2AVGSATMATH',
    'C2AVGSATREAD',
    'C2AVGSATWRIT',
]
# Keep only columns that actually exist in the loaded dataset
cleaned_continuous_cols_w2 = [c for c in cleaned_continuous_cols_w2 if c in wave2.columns]

cleaned_ordinal_cols_w2 = [
    'X2PAREDEXPCT',
    'X2STUEDEXPCT',
    'S2GRD1011',
    'S2GRD1112',
    'X2REQLEVEL',
    'S2SUREDIPL',
    'S2ALG1GRADE',
    'S2EDUEXP',
    'C2PCTEXAMINFO',
    'C2PCTINFO',
    'C2UPMCLGREQ',
    'C2UPMEOGEXAM',
    'C2UPMGRADREQ',
    'C2PCTEXAMPREP',
    'S2PAYOFF',
    'S2DOOKAY',
]
cleaned_ordinal_cols_w2 = [c for c in cleaned_ordinal_cols_w2 if c in wave2.columns]

cleaned_binary_cols_w2 = [
    'X2POVERTY',
    'C2AIDFLYER',
    'C2GRADPLAN',
    'C2INFOSESSN',
    'C2SELECTCLG',
    'C2CLGAPP',
    'C2INFSTEM',
    'S2ENROLLHS12',
]
cleaned_binary_cols_w2 = [c for c in cleaned_binary_cols_w2 if c in wave2.columns]

# Nominal columns (still in integer form in the EDA-ready CSV — encoding happens below)
nominal_cols_w2 = ['S2REQTYP4YR', 'S2CLG2013', 'S2MOSTIMP2013']
nominal_cols_w2 = [c for c in nominal_cols_w2 if c in wave2.columns]

print(f"\nColumn lists rebuilt:")
print(f"  Continuous : {len(cleaned_continuous_cols_w2)}")
print(f"  Ordinal    : {len(cleaned_ordinal_cols_w2)}")
print(f"  Binary     : {len(cleaned_binary_cols_w2)}")
print(f"  Nominal    : {len(nominal_cols_w2)} (to be encoded next)")
print(f"\nAll expected columns present in wave2: ",
      all(c in wave2.columns for c in
          cleaned_continuous_cols_w2 + cleaned_ordinal_cols_w2 +
          cleaned_binary_cols_w2 + nominal_cols_w2))


Loaded Wave2_EDA_Ready.csv  — shape: (15236, 91)
Loaded Wave1_FeatureSelected.csv — shape: (15900, 24)

Column lists rebuilt:
  Continuous : 13
  Ordinal    : 16
  Binary     : 8
  Nominal    : 3 (to be encoded next)

All expected columns present in wave2:  True


# Section 2- One-Hot Encode Nominal Variables

The three nominal columns are one-hot encoded for use in machine learning models. Before encoding, a missingness flag is created to preserve the information that certain rows had unit non-response (-8) across all three columns.

In [ ]:
nom_cols = ['S2REQTYP4YR', 'S2CLG2013', 'S2MOSTIMP2013']

# Create the unit non-response flag before touching values
# All three columns share the same -8 rows, so one flag captures the pattern
wave2['Skipped_REQTYP_CLG_MOSTIMP'] = (wave2[nom_cols[0]] == -8).astype(int)
print(f"Missingness flag created: Skipped_REQTYP_CLG_MOSTIMP")
print(f"  Flagged rows: {wave2['Skipped_REQTYP_CLG_MOSTIMP'].sum()}")

# Replace all sentinels with NaN
wave2[nom_cols] = wave2[nom_cols].replace({-8: np.nan, -9: np.nan})

# Impute with mode (all NaN rows — both -8 and -9 — receive the mode value)
# The -8 rows are already captured by the flag column
for col in nom_cols:
    mode_val    = wave2[col].mode()[0]
    wave2[col]  = wave2[col].fillna(mode_val)

# Cast to int then string so get_dummies produces clean column names
for col in nom_cols:
    wave2[col] = wave2[col].astype(int).astype(str)

# One-hot encode
wave2 = pd.get_dummies(wave2, columns=nom_cols, drop_first=False, dtype=int)

new_nominal_cols = (
    ['Skipped_REQTYP_CLG_MOSTIMP'] +
    [c for c in wave2.columns if any(c.startswith(n + '_') for n in nom_cols)]
)
print(f"\nNew columns from encoding ({len(new_nominal_cols)}): {new_nominal_cols}")
print(f"Dataset shape after encoding: {wave2.shape}")

Missingness flag created: Skipped_REQTYP_CLG_MOSTIMP
  Flagged rows: 920

New columns from encoding (12): ['Skipped_REQTYP_CLG_MOSTIMP', 'S2REQTYP4YR_1', 'S2REQTYP4YR_2', 'S2REQTYP4YR_3', 'S2CLG2013_1', 'S2CLG2013_2', 'S2CLG2013_3', 'S2MOSTIMP2013_1', 'S2MOSTIMP2013_2', 'S2MOSTIMP2013_3', 'S2MOSTIMP2013_4', 'S2MOSTIMP2013_5']
Dataset shape after encoding: (15236, 100)


#Section 3- Drop STU_ID from the Encoded Dataset

In [ ]:
wave2 = wave2.drop(columns=['STU_ID'], errors='ignore')
print(f"Dropped STU_ID. Shape: {wave2.shape}")

Dropped STU_ID. Shape: (15236, 99)


---
# Section 4 — Export Datasets

Four datasets are produced from the cleaned and encoded data. Each serves a different modeling purpose.

## 4.1 Define Column Sets

In [ ]:
# All cleaned Wave 1 columns (excluding STU_ID)
wave1_cols = [
    'X4EVERDROP',
    'X1MTHEFF', 'X1SCIEFF', 'X1TXMTSCOR',
    'S1M8GRADE', 'S1S8GRADE',
    'X1PAREDU', 'X1PAREDEXPCT', 'X1POVERTY', 'X1FAMINCOME',
    'S1EDUEXPECT', 'S1SUREHSGRAD',
    'S1PAYOFF', 'S1GETINTOCLG', 'S1AFFORD', 'S1WORKING',
    'X1SCHOOLBEL', 'X1SCHOOLCLI',
    'X2SEX', 'X1CONTROL',
    'S1FYAA', 'S1FYBA', 'S1FYLICENSE', 'S1FYAPPR',
    'S1FYMILITARY', 'S1FYJOB', 'S1FYFAMILY',
    'S1FYTRAVEL', 'S1FYVOLUN', 'S1FYNOTSURE',
    'X1PAR_SURVEY_MISSING', 'X1_DUAL_PARENT',
    'MATH_NO_CLASS', 'SCI_NO_CLASS',
    'X1RACE_1', 'X1RACE_2', 'X1RACE_3', 'X1RACE_4',
    'X1RACE_5', 'X1RACE_6', 'X1RACE_7', 'X1RACE_8',
    'X1LOCALE_1', 'X1LOCALE_2', 'X1LOCALE_3', 'X1LOCALE_4',
    'X1REGION_1', 'X1REGION_2', 'X1REGION_3', 'X1REGION_4',
]

# All cleaned Wave 2 columns (continuous + ordinal + binary + nominal encoded)
nominal_encoded_cols = (
    ['Skipped_REQTYP_CLG_MOSTIMP'] +
    [c for c in wave2.columns if any(c.startswith(n + '_') for n in
     ['S2REQTYP4YR', 'S2CLG2013', 'S2MOSTIMP2013'])]
)

all_cleaned_wave2_cols = (
    cleaned_continuous_cols_w2 +
    cleaned_ordinal_cols_w2 +
    cleaned_binary_cols_w2 +
    nominal_encoded_cols
)

# EDA-justified selected Wave 2 features
# Removed: S2EDUEXP (redundant with X2STUEDEXPCT), S2ENROLLHS12 (target leakage)
# See justification in section header
final_wave2_features = [
    "X2MTHEFF",       # Math self-efficacy (Wave 2)
    "X2SCIEFF",       # Science self-efficacy (Wave 2)
    "X2TXMTSCOR",     # Math test score (Wave 2) — strong predictor
    "X2BEHAVEIN",     # School motivation/behavior — strong predictor
    "X2PAREDEXPCT",   # Parent educational expectation (Wave 2)
    "X2STUEDEXPCT",   # Student educational expectation (Wave 2)
    "S2GRD1011",      # Grade level 2010-2011
    "S2GRD1112",      # Grade level 2011-2012
    "X2REQLEVEL",     # Highest level meeting college requirements — strong predictor
    "S2SUREDIPL",     # Certainty of HS diploma — strong predictor
    "S2ALG1GRADE",    # Algebra I final grade — strong predictor
    "S2PAYOFF",       # Studying pays off belief (Wave 2)
    "S2DOOKAY",       # Can do okay as dropout belief
    "X2POVERTY",      # Poverty status (Wave 2) — strong predictor
]

print(f"Wave 1 columns         : {len(wave1_cols)}")
print(f"All Wave 2 columns     : {len(all_cleaned_wave2_cols)}")
print(f"Selected Wave 2 features: {len(final_wave2_features)}")

Wave 1 columns         : 50
All Wave 2 columns     : 49
Selected Wave 2 features: 14


## 4.2 Feature Selection Justification — Wave 2 Variables

The following Wave 2 features are excluded from the enhanced model datasets based on findings from EDA:

| Feature | Reason for Exclusion |
|---------|----------------------|
| `S2EDUEXP` | Highly redundant with `X2STUEDEXPCT` — both measure student educational expectation in 11th grade. Retaining both would introduce multicollinearity without adding predictive value. |
| `S2ENROLLHS12` | Records whether the student was enrolled in spring 2012. This variable is derived from the same outcome being predicted and would introduce target leakage. |
| Helper columns (e.g., `EXPCT_GRP`) | Temporary grouping variables created for visualization during EDA. They carry no independent information and must be removed before model training. |

The following Wave 2 features are highlighted from EDA as especially important predictors and are retained in all model datasets:

`X2TXMTSCOR`, `X2REQLEVEL`, `S2SUREDIPL`, `S2ALG1GRADE`, `X2BEHAVEIN`, `X2POVERTY`

No dimensionality reduction is applied. The feature set is already compact and interpretability is more valuable than further compression at this stage.

## 4.3 Dataset 1 — Baseline: All Wave 1 + All Wave 2

All cleaned Wave 1 columns combined with all cleaned Wave 2 columns. This is the most complete dataset and is used for the baseline model comparison.

In [ ]:
cleaned_wave2_no_target = [c for c in all_cleaned_wave2_cols if c != 'X4EVERDROP']

dataset_baseline = wave2[wave1_cols + cleaned_wave2_no_target].copy()

print("=== Baseline Dataset: All Wave 1 + All Wave 2 ===")
print(f"  Shape    : {dataset_baseline.shape}")
print(f"  Target   : X4EVERDROP")
print(f"  Features : {len(wave1_cols) + len(cleaned_wave2_no_target) - 1}")
print(f"  NaN      : {dataset_baseline.isnull().sum().sum()}")

dataset_baseline.to_csv("Baseline_Wave1_Wave2.csv", index=False)
print("  Saved    : Baseline_Wave1_Wave2.csv")

=== Baseline Dataset: All Wave 1 + All Wave 2 ===
  Shape    : (15236, 99)
  Target   : X4EVERDROP
  Features : 98
  NaN      : 0
  Saved    : Baseline_Wave1_Wave2.csv


## 4.4 Dataset 2 — Enhanced Model V1: All Wave 1 + Selected Wave 2

All Wave 1 columns combined with the EDA-justified selected Wave 2 features.

In [ ]:
wave1_cols_no_target = [c for c in wave1_cols if c != 'X4EVERDROP']

dataset_enhanced_v1 = wave2[['X4EVERDROP'] + wave1_cols_no_target + final_wave2_features].copy()

print("=== Enhanced Model V1: All Wave 1 + Selected Wave 2 ===")
print(f"  Shape    : {dataset_enhanced_v1.shape}")
print(f"  Target   : X4EVERDROP")
print(f"  Features : {len(wave1_cols_no_target) + len(final_wave2_features)}")
print(f"  NaN      : {dataset_enhanced_v1.isnull().sum().sum()}")

dataset_enhanced_v1.to_csv("Enhanced_Model_V1.csv", index=False)
print("  Saved    : Enhanced_Model_V1.csv")

=== Enhanced Model V1: All Wave 1 + Selected Wave 2 ===
  Shape    : (15236, 64)
  Target   : X4EVERDROP
  Features : 63
  NaN      : 0
  Saved    : Enhanced_Model_V1.csv


## 4.5 Dataset 3 — Enhanced Model V2: Selected Wave 1 + Selected Wave 2

The reduced Wave 1 feature set from `Wave1_FeatureSelected.csv` combined with the selected Wave 2 features. This dataset uses the most carefully curated predictors from both waves.

In [ ]:
# Merge selected Wave 1 features with the selected Wave 2 features
# wave1_selected has STU_ID which we use to align rows, then drop
wave1_sel_cols = [c for c in wave1_selected.columns if c != 'STU_ID']                   if 'STU_ID' in wave1_selected.columns else wave1_selected.columns.tolist()

# Build the V2 dataset using the wave2 dataframe (already contains Wave 1 cols via merge)
# Pull exactly the columns from the wave1 feature selection, plus the selected wave2 features
wave1_final_feature_cols = [c for c in wave1_sel_cols if c in wave2.columns and c != 'X4EVERDROP']

dataset_enhanced_v2 = wave2[['X4EVERDROP'] + wave1_final_feature_cols + final_wave2_features].copy()

print("=== Enhanced Model V2: Selected Wave 1 + Selected Wave 2 ===")
print(f"  Shape    : {dataset_enhanced_v2.shape}")
print(f"  Target   : X4EVERDROP")
print(f"  Wave 1 features : {len(wave1_final_feature_cols)}")
print(f"  Wave 2 features : {len(final_wave2_features)}")
print(f"  NaN      : {dataset_enhanced_v2.isnull().sum().sum()}")

dataset_enhanced_v2.to_csv("Enhanced_Model_V2.csv", index=False)
print("  Saved    : Enhanced_Model_V2.csv")

=== Enhanced Model V2: Selected Wave 1 + Selected Wave 2 ===
  Shape    : (15236, 38)
  Target   : X4EVERDROP
  Wave 1 features : 23
  Wave 2 features : 14
  NaN      : 0
  Saved    : Enhanced_Model_V2.csv


## 4.6 Final Summary

All four output datasets are ready for use.

In [ ]:
print("=" * 65)
print("  OUTPUT DATASETS SUMMARY")
print("=" * 65)
print(f"  {'File':<38} {'Shape'}")
print(f"  {'─'*38} {'─'*15}")

print(f"  {'Baseline_Wave1_Wave2.csv':<38} {str(dataset_baseline.shape)}")
print(f"  {'Enhanced_Model_V1.csv':<38} {str(dataset_enhanced_v1.shape)}")
print(f"  {'Enhanced_Model_V2.csv':<38} {str(dataset_enhanced_v2.shape)}")
print()
print("  Notes:")

print("  - Baseline          : all Wave 1 + all cleaned Wave 2 columns")
print("  - Enhanced V1       : all Wave 1 + EDA-selected Wave 2 features")
print("  - Enhanced V2       : feature-selected Wave 1 + EDA-selected Wave 2 features")
print()


  OUTPUT DATASETS SUMMARY
  File                                   Shape
  ────────────────────────────────────── ───────────────
  Baseline_Wave1_Wave2.csv               (15236, 99)
  Enhanced_Model_V1.csv                  (15236, 64)
  Enhanced_Model_V2.csv                  (15236, 38)

  Notes:
  - Baseline          : all Wave 1 + all cleaned Wave 2 columns
  - Enhanced V1       : all Wave 1 + EDA-selected Wave 2 features
  - Enhanced V2       : feature-selected Wave 1 + EDA-selected Wave 2 features

